# DistilBERT Email Security Classifier

Notebook này train classifier 3 lớp cho project `spam-agent`:

```text
safe | phishing | spam
```

Dataset chính: Kaggle `akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000`.

Pipeline:

```text
load dataset -> clean text/label -> stratified train/valid/test split -> class-weighted DistilBERT training -> evaluate -> save artifact
```

## 1. Install Dependencies

Chạy cell này trên Kaggle. Không cài lại `torch` vì Kaggle đã có bản PyTorch CUDA phù hợp với GPU runtime.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy

## 2. Imports And Config

In [ ]:
from pathlib import Path
import inspect
import json
import os
import random
import zipfile

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. On Kaggle, open Settings > Accelerator and select GPU, then restart the session."
    )

DEVICE = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

BASE_MODEL = "distilbert-base-uncased"
OUTPUT_DIR = Path("/kaggle/working/distilbert_email_security")
MAX_LENGTH = 512
EPOCHS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5

LABEL2ID = {"safe": 0, "phishing": 1, "spam": 2}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

LABEL_ALIASES = {
    "0": "safe",
    "ham": "safe",
    "safe": "safe",
    "legitimate": "safe",
    "not spam": "safe",
    "non-spam": "safe",
    "1": "phishing",
    "phish": "phishing",
    "phishing": "phishing",
    "scam": "phishing",
    "fraud": "phishing",
    "2": "spam",
    "spam": "spam",
    "true": "spam",
}

print("cuda_available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))

## 3. Locate Or Download Dataset

Cách dễ nhất trên Kaggle là dùng `Add Input` và thêm dataset:

`akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000`

Nếu chưa add input, cell dưới sẽ thử tải bằng Kaggle API.

In [ ]:
dataset_input = Path("/kaggle/input/the-biggest-spam-ham-phish-email-dataset-300000")
dataset_working = Path("/kaggle/working/dataset")

if not dataset_input.exists() and not dataset_working.exists():
    !kaggle datasets download -d akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000 -p /kaggle/working/dataset --unzip

input_dir = dataset_input if dataset_input.exists() else dataset_working
csv_files = sorted(input_dir.glob("*.csv"))
print(input_dir)
print(csv_files)
assert csv_files, f"No CSV files found in {input_dir}"

## 4. Load And Normalize Columns

In [ ]:
raw_df = pd.read_csv(csv_files[0])
raw_df.columns = [c.strip().lower() for c in raw_df.columns]
print(raw_df.shape)
print(raw_df.columns.tolist())
raw_df.head()

In [ ]:
text_candidates = {"text", "email", "message", "body", "content"}
label_candidates = {"label", "class", "category", "target"}

text_col = next(c for c in raw_df.columns if c in text_candidates)
label_col = next(c for c in raw_df.columns if c in label_candidates)
print({"text_col": text_col, "label_col": label_col})

df = raw_df.rename(columns={text_col: "text", label_col: "label"})[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str)
df["label"] = df["label"].astype(str).str.lower().str.strip().map(LABEL_ALIASES)

df = df.dropna(subset=["label"])
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df["text"].str.len() >= 20]
df = df.drop_duplicates(subset=["text"])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts())
df.head()

## 5. Stratified Train / Validation / Test Split

Validation và test giữ phân phối thật. Không oversample hai split này.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED,
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=SEED,
)

for name, split in {"train": train_df, "valid": valid_df, "test": test_df}.items():
    print(name, split.shape)
    print(split["label"].value_counts())
    split.to_csv(f"/kaggle/working/{name}.csv", index=False)

## 6. Optional: Oversample Phishing In Train Only

Mặc định dùng class weights là chính. Chỉ bật oversampling nếu lần train baseline có `phishing_recall` thấp.

In [ ]:
USE_OVERSAMPLED_TRAIN = False
TARGET_PHISHING = 85000

if USE_OVERSAMPLED_TRAIN:
    phishing_df = train_df[train_df["label"] == "phishing"]
    other_df = train_df[train_df["label"] != "phishing"]
    phishing_up = phishing_df.sample(n=TARGET_PHISHING, replace=True, random_state=SEED)
    train_model_df = pd.concat([other_df, phishing_up], ignore_index=True).sample(frac=1, random_state=SEED)
else:
    train_model_df = train_df.copy()

train_model_df = train_model_df.reset_index(drop=True)
train_model_df.to_csv("/kaggle/working/train_model.csv", index=False)
print(train_model_df["label"].value_counts())

## 7. Compute Class Weights

Class weights được tính từ train set dùng để train model. Với phân phối mẫu thường gặp, phishing sẽ có weight cao hơn.

In [ ]:
train_labels = train_model_df["label"].map(LABEL2ID).astype(int).tolist()
counts = np.bincount(train_labels, minlength=len(LABEL2ID))
total = counts.sum()
class_weights = [float(total / (len(LABEL2ID) * count)) if count else 0.0 for count in counts]

print({ID2LABEL[i]: class_weights[i] for i in range(len(class_weights))})

## 8. Build Hugging Face Datasets

In [ ]:
def to_hf_dataset(split_df: pd.DataFrame) -> Dataset:
    return Dataset.from_dict({
        "text": split_df["text"].astype(str).tolist(),
        "label": split_df["label"].map(LABEL2ID).astype(int).tolist(),
    })

dataset = {
    "train": to_hf_dataset(train_model_df),
    "validation": to_hf_dataset(valid_df),
    "test": to_hf_dataset(test_df),
}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = {name: split.map(tokenize, batched=True) for name, split in dataset.items()}

## 9. Metrics And Weighted Trainer

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1, 2],
        average="weighted",
        zero_division=0,
    )
    metrics = {
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "confusion_matrix": confusion_matrix(labels, preds, labels=[0, 1, 2]).tolist(),
    }
    for idx, label in ID2LABEL.items():
        metrics[f"{label}_precision"] = float(precision[idx])
        metrics[f"{label}_recall"] = float(recall[idx])
        metrics[f"{label}_f1"] = float(f1[idx])
    return metrics


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is None:
            loss = outputs.get("loss")
        else:
            weights = torch.tensor(self.class_weights, dtype=logits.dtype, device=logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
            loss = loss_fn(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 10. Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)


training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    dataloader_pin_memory=True,
    report_to="none",
)

if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["evaluation_strategy"] = "epoch"
else:
    training_kwargs["eval_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 11. Evaluate Validation And Test

In [ ]:
validation_metrics = trainer.evaluate(eval_dataset=tokenized["validation"])
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"], metric_key_prefix="test")

payload = {
    "base_model": BASE_MODEL,
    "labels": LABEL2ID,
    "class_weights": class_weights,
    "use_oversampled_train": USE_OVERSAMPLED_TRAIN,
    "validation": validation_metrics,
    "test": test_metrics,
}

print(json.dumps(payload, indent=2))

## 12. Save Model Artifact

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 13. Quick Inference Smoke Test

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(OUTPUT_DIR),
    tokenizer=str(OUTPUT_DIR),
    truncation=True,
    max_length=MAX_LENGTH,
    top_k=None,
)

samples = [
    "Please review the meeting notes and send feedback by Friday.",
    "Congratulations, you won a free gift card. Claim your prize now.",
    "Your account will be suspended. Verify your password at http://login-example.xyz immediately.",
]

for text in samples:
    print(text)
    print(clf(text))

## 14. Zip Artifact For Download

Sau khi tải về máy, giải nén nội dung vào project:

```text
models/distilbert_multilingual/
```

In [ ]:
zip_path = Path("/kaggle/working/distilbert_email_security.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(Path("distilbert_email_security") / path.relative_to(OUTPUT_DIR)))

print(zip_path)
print(zip_path.stat().st_size / (1024 * 1024), "MB")